# Quick Load and Predict

Minimal notebook: load the trained model, load one degraded image, run
prediction, view the result. For a fuller test suite (batch processing,
quantitative metrics, multiple examples), see `Proposed_Model_Testing.ipynb`
instead — this notebook is intentionally the smallest possible working example.

In [ ]:
import sys, os
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt

from models.restoration_net import DistributionMixtureRestorationNet


## 1. Load the trained model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

CHECKPOINT_PATH = "../DistributionMixtureRestorationNet.pth"

model = DistributionMixtureRestorationNet(base_ch=32, n_components=3,
                                            n_lr_blocks=4, n_hr_blocks=2, use_film=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device).eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation PSNR: {checkpoint['best_val_psnr']:.3f}")
print(f"Best validation SSIM: {checkpoint['best_val_ssim']:.4f}")


## 2. Load one degraded image

Set `IMAGE_PATH` to a real `.npy` file (128x128, float32) before running.

In [ ]:
IMAGE_PATH = r"path\to\one\degraded_image.npy"   # <-- CHANGE THIS

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(
        f"{IMAGE_PATH} does not exist -- set IMAGE_PATH to a real degraded .npy file before continuing."
    )

degraded = np.load(IMAGE_PATH).astype(np.float32)
print("Loaded image shape:", degraded.shape, " dtype:", degraded.dtype)


## 3. Predict (restore)

In [ ]:
with torch.no_grad():
    x = torch.from_numpy(degraded).unsqueeze(0).unsqueeze(0).to(device)  # [1, 1, H, W]
    restored, mix_weights, beta, scale = model(x)
    restored_np = restored[0, 0].clamp(0, 1).cpu().numpy()

print("Input shape: ", degraded.shape)
print("Output shape:", restored_np.shape)
print("Output range: [{:.4f}, {:.4f}]".format(restored_np.min(), restored_np.max()))


## 4. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(degraded, cmap="gray")
axes[0].set_title("Degraded (input)")
axes[0].axis("off")

axes[1].imshow(restored_np, cmap="gray")
axes[1].set_title("Restored (output)")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("../results/quick_predict_example.png", dpi=150)
plt.show()
print("Saved: ../results/quick_predict_example.png")


## 5. (Optional) Save the restored image

Uncomment and run to save the output as its own `.npy` file.

In [ ]:
# out_path = "../results/quick_predict_output.npy"
# np.save(out_path, restored_np.astype(np.float32))
# print("Saved:", out_path)
